# Debug Tokenizer

Notebook nay dung de kiem tra day du `modules/tokenizers.py`: load annotation, clean report, tao vocabulary, encode/decode, UNK ratio, do dai chuoi va cac edge cases thuong gap.

Chay tu tren xuong duoi. Neu muon doi dataset hoac threshold, chi can sua o phan **Config**.

## 1. Setup moi truong

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from collections import Counter
import json
import sys

try:
    import pandas as pd
except ImportError:
    pd = None

project_root = Path.cwd().resolve()
if not (project_root / 'modules').exists():
    project_root = project_root.parent.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from modules.tokenizers import Tokenizer

def show_table(rows, columns=None, max_rows=20):
    rows = list(rows)
    if pd is not None:
        return pd.DataFrame(rows, columns=columns).head(max_rows)
    for row in rows[:max_rows]:
        print(row)

print('Project root:', project_root)
print('Pandas available:', pd is not None)

## 2. Config

`Tokenizer` can 3 tham so: `ann_path`, `dataset_name`, `threshold`. Mac dinh cua repo la IU X-Ray voi threshold = 3.

In [ ]:
DATASET_NAME = 'iu_xray'      # 'iu_xray' hoac 'mimic_cxr'
THRESHOLD = 3                 # iu_xray thuong dung 3, mimic_cxr thuong dung 10
MAX_SEQ_LENGTH = 60           # dung de debug truncate nhu Dataset trong modules/datasets.py
SAMPLE_INDEX = 0              # doi index de xem report khac trong split train

DATASET_CONFIG = {
    'iu_xray': project_root / 'data' / 'iu_xray' / 'annotation.json',
    'mimic_cxr': project_root / 'data' / 'mimic_cxr' / 'annotation.json',
}

ann_path = DATASET_CONFIG[DATASET_NAME]
args = SimpleNamespace(
    ann_path=str(ann_path),
    dataset_name=DATASET_NAME,
    threshold=THRESHOLD,
)

print('Dataset      :', args.dataset_name)
print('Threshold    :', args.threshold)
print('Annotation   :', Path(args.ann_path))
print('File exists  :', Path(args.ann_path).exists())

## 3. Kiem tra annotation

Buoc nay giup phat hien som loi sai path, thieu split, thieu field `report`, hoac annotation rong.

In [ ]:
assert Path(args.ann_path).exists(), f'Khong tim thay annotation: {args.ann_path}'

ann = json.loads(Path(args.ann_path).read_text(encoding='utf-8'))
required_splits = ['train', 'val', 'test']

for split in required_splits:
    assert split in ann, f'Missing split: {split}'
    assert isinstance(ann[split], list), f'Split {split} phai la list'
    assert len(ann[split]) > 0, f'Split {split} dang rong'
    assert 'report' in ann[split][0], f'Missing field report trong split {split}'

split_rows = []
for split in required_splits:
    split_rows.append({
        'split': split,
        'num_examples': len(ann[split]),
        'first_id': ann[split][0].get('id', '<missing>'),
        'first_report_chars': len(ann[split][0]['report']),
    })

show_table(split_rows)

## 4. Khoi tao Tokenizer va vocab

In [ ]:
tokenizer = Tokenizer(args)

print('Vocab size  :', tokenizer.get_vocab_size())
print('PAD id      :', tokenizer.pad_idx)
print('BOS id      :', tokenizer.bos_idx)
print('EOS id      :', tokenizer.eos_idx)
print('UNK id      :', tokenizer.unk_idx)
print('Decode note : decode(ids) tu bo qua BOS/PAD va dung tai EOS')

vocab_preview = [
    {'idx': idx, 'token': token}
    for token, idx in list(tokenizer.token2idx.items())[:40]
]
show_table(vocab_preview)

## 5. Tan suat token va tac dong cua threshold

`create_vocabulary()` chi dung split `train`. Cac token co tan suat nho hon `threshold` se map ve `<unk>`.

In [ ]:
def clean_tokens(report):
    return tokenizer.clean_report(report).split()

train_tokens = []
for ex in ann['train']:
    train_tokens.extend(clean_tokens(ex['report']))

counter = Counter(train_tokens)
freq_rows = [
    {
        'token': token,
        'freq': freq,
        'in_vocab': token in tokenizer.token2idx,
        'id': tokenizer.token2idx.get(token, tokenizer.get_id_by_token('<unk>')),
    }
    for token, freq in counter.most_common(40)
]

print('Total train tokens :', len(train_tokens))
print('Unique train tokens:', len(counter))
show_table(freq_rows)

In [ ]:
threshold_rows = []
for threshold in [1, 2, 3, 5, 10, 20]:
    kept = sum(1 for freq in counter.values() if freq >= threshold)
    dropped = len(counter) - kept
    threshold_rows.append({
        'threshold': threshold,
        'vocab_size_including_specials': kept + 4,
        'dropped_unique_tokens': dropped,
        'dropped_ratio_unique': f'{dropped / max(len(counter), 1):.2%}',
    })

show_table(threshold_rows)

## 6. Debug mot report cu the

O nay hien raw text, cleaned text, ids va bang token-by-token de xem token nao bi UNK.

In [ ]:
def inspect_report(report, title='report'):
    cleaned = tokenizer.clean_report(report)
    tokens = cleaned.split()
    ids = tokenizer(report)
    unk_id = tokenizer.get_id_by_token('<unk>')

    print('=' * 100)
    print(title)
    print('- raw')
    print(report)
    print('\n- cleaned')
    print(cleaned)
    print('\n- ids')
    print(ids)
    print('\n- decoded')
    print(tokenizer.decode(ids))

    rows = []
    for pos, token in enumerate(tokens, start=1):
        idx = tokenizer.get_id_by_token(token)
        rows.append({
            'pos': pos,
            'token': token,
            'id': idx,
            'is_unk': idx == unk_id,
            'train_freq': counter.get(token, 0),
        })
    return show_table(rows, max_rows=200)

sample = ann['train'][SAMPLE_INDEX]
inspect_report(sample['report'], title=f"train[{SAMPLE_INDEX}] id={sample.get('id', '<missing>')}")

## 7. Edge cases

Dung de xem tokenizer xu ly chuoi rong, dau cau, numbering va token ngoai vocab nhu the nao.

In [ ]:
edge_cases = [
    '',
    '   ',
    'No acute cardiopulmonary abnormality.',
    'Heart size is normal..  1. No focal consolidation. 2. No pleural effusion.',
    'Lines/tubes: ET-tube is 3.5 cm above carina; lungs are clear!',
    'UNSEEN_TOKEN_ABC xyzzy plugh',
]

edge_rows = []
for text in edge_cases:
    cleaned = tokenizer.clean_report(text)
    ids = tokenizer(text)
    edge_rows.append({
        'input': repr(text),
        'cleaned': cleaned,
        'num_tokens': len(cleaned.split()),
        'ids': ids,
        'decoded': tokenizer.decode(ids),
    })

show_table(edge_rows, max_rows=20)

## 8. UNK ratio theo split

Neu UNK ratio cao, co the threshold dang qua lon hoac clean text dang tao ra qua nhieu bien the token.

In [ ]:
def unk_stats_for_examples(examples):
    unk_id = tokenizer.get_id_by_token('<unk>')
    total = 0
    unk = 0
    unk_tokens = Counter()

    for ex in examples:
        for token in clean_tokens(ex['report']):
            total += 1
            if tokenizer.get_id_by_token(token) == unk_id:
                unk += 1
                unk_tokens[token] += 1

    return total, unk, unk / max(total, 1), unk_tokens

unk_rows = []
all_unk_tokens = Counter()
for split in required_splits:
    total, unk, ratio, unk_tokens = unk_stats_for_examples(ann[split])
    all_unk_tokens.update(unk_tokens)
    unk_rows.append({
        'split': split,
        'tokens': total,
        'unk_tokens': unk,
        'unk_ratio': f'{ratio:.2%}',
    })

display_obj = show_table(unk_rows)
display_obj

In [ ]:
top_unk_rows = [
    {'token': token, 'count': count}
    for token, count in all_unk_tokens.most_common(40)
]

print('Top UNK tokens across splits')
show_table(top_unk_rows)

## 9. Do dai sequence va truncate

`BaseDataset` cat ids bang `tokenizer(report)[:max_seq_length]`. Neu cat qua ngan co the mat token `<eos>` o cuoi report.

In [ ]:
def seq_len(report):
    return len(tokenizer(report))

length_rows = []
long_examples = []
for split in required_splits:
    lengths = [seq_len(ex['report']) for ex in ann[split]]
    truncated = sum(length > MAX_SEQ_LENGTH for length in lengths)
    length_rows.append({
        'split': split,
        'min': min(lengths),
        'avg': round(sum(lengths) / len(lengths), 2),
        'max': max(lengths),
        f'> {MAX_SEQ_LENGTH}': truncated,
        'truncate_ratio': f'{truncated / len(lengths):.2%}',
    })
    for ex, length in zip(ann[split], lengths):
        if length > MAX_SEQ_LENGTH:
            long_examples.append({'split': split, 'id': ex.get('id'), 'seq_len': length, 'report': ex['report']})

show_table(length_rows)

In [ ]:
print(f'Examples bi cat neu MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}:', len(long_examples))
for ex in long_examples[:3]:
    ids = tokenizer(ex['report'])
    truncated_ids = ids[:MAX_SEQ_LENGTH]
    print('=' * 100)
    print('split:', ex['split'], '| id:', ex['id'], '| original_len:', ex['seq_len'], '| truncated_len:', len(truncated_ids))
    print('last original ids :', ids[-10:])
    print('last truncated ids:', truncated_ids[-10:])
    print('truncated decoded :', tokenizer.decode(truncated_ids))

## 10. Sanity checks

Cac assert nay khong thay the unit test, nhung giup phat hien nhanh loi nghiem trong khi sua tokenizer.

In [ ]:
unk_id = tokenizer.get_id_by_token('<unk>')

assert tokenizer.get_vocab_size() == len(tokenizer.token2idx) == len(tokenizer.idx2token)
assert '<unk>' in tokenizer.token2idx
assert tokenizer.idx2token[unk_id] == '<unk>'

for ex in ann['train'][:50]:
    ids = tokenizer(ex['report'])
    assert ids[0] == tokenizer.bos_idx, 'ids phai bat dau bang BOS'
    assert ids[-1] == tokenizer.eos_idx, 'ids phai ket thuc bang EOS'
    assert all(isinstance(x, int) for x in ids), 'ids phai la int'
    decoded = tokenizer.decode(ids)
    assert isinstance(decoded, str)

print('All sanity checks passed.')

## Checklist khi debug

- Sai path annotation: kiem tra `ann_path` trong Config.
- Vocab qua nho hoac UNK ratio cao: thu giam `THRESHOLD`.
- Decode bi cat som: kiem tra vi tri token `<eos>` trong sequence.
- Mat EOS sau truncate: tang `MAX_SEQ_LENGTH` hoac xu ly cat chuoi de giu token EOS.
- Clean text khong nhu mong muon: xem lai `clean_report_iu_xray()` va `clean_report_mimic_cxr()` trong `modules/tokenizers.py`.